# Phylogenetic Generalized Least Squares (PGLS)

In [35]:

import numpy as np
import toytree

## Simulate data

In [36]:
tree = toytree.rtree.unittree(ntips=50, treeheight=1.0, seed=123)

In [37]:
traits = toytree.pcm.simulate_continuous_bm(tree, sigma2=[0.1, 0.01], root_state=[5.0, 0.0], tips_only=True)

In [38]:
fixed = np.array([1] * tree.ntips)
fixed[10:15] = 0
fixed

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1])

In [39]:
tree.write("/tmp/test.nwk")
traits.to_csv("/tmp/test.csv")

## Analyze

In [40]:
from toytree.pcm.src.traits.pgls_matrix import pgls_matrix

In [41]:
model = pgls_matrix(tree, "t0 ~ t1", traits)
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            GLS Regression Results                            
==============================================================================
Dep. Variable:                     t0   R-squared:                       0.025
Model:                            GLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     1.211
Date:                Sun, 16 Feb 2025   Prob (F-statistic):              0.277
Time:                        12:15:46   Log-Likelihood:                 81.606
No. Observations:                  50   AIC:                            -159.2
Df Residuals:                      48   BIC:                            -155.4
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      5.0127      0.022    231.808      0.000       4.969       5.056
t1            -1.8382      1.671     -1.100      0.277      -5.197       1.521
==============================================================================
Omnibus:                        3.004   Durbin-Watson:                   2.030
Prob(Omnibus):                  0.223   Jarque-Bera (JB):                1.997
Skew:                          -0.413   Prob(JB):                        0.368
Kurtosis:                       3.525   Cond. No.                         77.3
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [42]:
from statsmodels.api import MixedLM

In [46]:
model = MixedLM.from_formula("t0 ~ t1", data=traits, groups=fixed, re_formula="1 + t1")
fit = model.fit(reml=True)
fit.summary()

/home/deren/miniconda3/envs/toytree-dev/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/deren/miniconda3/envs/toytree-dev/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
/home/deren/miniconda3/envs/toytree-dev/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/home/deren/miniconda3/envs/toytree-dev/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/home/deren/miniconda3/envs/toytree-dev/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2261: Con

<class 'statsmodels.iolib.summary2.Summary'>
"""
        Mixed Linear Model Regression Results
=====================================================
Model:            MixedLM Dependent Variable: t0     
No. Observations: 50      Method:             REML   
No. Groups:       2       Scale:              0.0027 
Min. group size:  5       Log-Likelihood:     74.9519
Max. group size:  45      Converged:          Yes    
Mean group size:  25.0                               
-----------------------------------------------------
                Coef.  Std.Err. z P>|z| [0.025 0.975]
-----------------------------------------------------
Intercept        5.002                               
t1              -1.203                               
Group Var        0.000                               
Group x t1 Cov  -0.000                               
t1 Var           0.003                               
=====================================================

"""

## R code

In [9]:
# Generalized least squares fit by REML
#   Model: t0 ~ t1
#   Data: traits
#         AIC       BIC   logLik
#   -166.7647 -161.1511 86.38237

# Correlation Structure: corBrownian
#  Formula: ~1
#  Parameter estimate(s):
# numeric(0)

# Coefficients:
#                Value Std.Error   t-value p-value
# (Intercept) 4.995120 0.0189004 264.28595  0.0000
# t1          0.734132 1.1216254   0.65453  0.5159

#  Correlation:
#    (Intr)
# t1 -0.069

# Standardized residuals:
#        Min         Q1        Med         Q3        Max
# -2.4537513 -0.5680907 -0.3333414  0.1460326  2.8324501

# Residual standard error: 0.05572073
# Degrees of freedom: 50 total; 48 residual

In [160]:
t = toytree.rtree.unittree(10, seed=123)

In [161]:
c = t.copy()

# ...
tip_to_root_dists1 = c.distance.get_node_distance_matrix()[-1]
#print(tip_to_root_dists1)

# multiply internal edges by lambda
lam = 1.25
for n in c[c.ntips:]:
    n._dist *= lam

tip_to_root_dists2 = c.distance.get_node_distance_matrix()[-1]
print(tip_to_root_dists1 - tip_to_root_dists2)

# extend tips to original distance from root
for n in c[:c.ntips]:
    n._dist += tip_to_root_dists1[n.idx] - tip_to_root_dists2[n.idx]
    #n._height = 0.
c._update()

#print(c.get_node_data("dist"))

toytree.mtree([t, c]).draw(ts='c', scale_bar=True);

[-0.15 -0.2  -0.2  -0.2  -0.2  -0.15 -0.1  -0.1  -0.05 -0.05 -0.2  -0.15
 -0.2  -0.15 -0.1  -0.1  -0.05 -0.05  0.  ]


<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="450.0px" height="250.0px" viewBox="0 0 450.0 250.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t60d7b2220e8f4344ad8294a5c9298e8f"> r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 0 0.5 1 r0 r1 r2 r3 r4 r5 r6 r7 r8 r9 0 0.5 1

In [144]:

    
max_lambda(t)

1.25

In [118]:
tip_to_root_dists1[c.ntips:] * x = 1.0

SyntaxError: cannot assign to expression here. Maybe you meant '==' instead of '='? (349444799.py, line 1)